# AI, Data Science and Software Development Employment – Visualisations

This notebook uses the **U.S. Bureau of Labor Statistics (BLS) Occupational Employment and Wage Statistics (OEWS)** "All data" files from https://www.bls.gov/oes/tables.htm to explore employment in jobs involving **AI, Data Science and Software Development**.

Dataset: the **May 2023** "All data" file (`all_data_M_2023.xlsx`), which must be in the same folder as this notebook.

For each visualisation, we explain:

1. The **objective** of the visualisation (what we want to see).
2. **Why that visualisation type was chosen** (categorical, numerical, relationships).
3. **What we learned** about AI employment from the plot.

**Which jobs count as "AI"?** The BLS has no job called "AI Engineer", so we use the nine occupations where AI work is done:

| Field | Occupations |
|---|---|
| **AI & Research** | Computer and Information Research Scientists (the people who invent new AI and machine-learning methods) |
| **Data Science** | Data Scientists, Statisticians, Operations Research Analysts, Mathematicians, Database Architects |
| **Software Development** | Software Developers, Software Quality Assurance Analysts and Testers, Web Developers |

⚠️ *Limitation:* many machine-learning engineers are counted as "Software Developers" or "Data Scientists", so these numbers approximate AI jobs and are not an exact count.

> **Version 01 of 10.** What changed in this version: Project set-up: introduction, loading the May 2023 BLS data, cleaning (`*`, `**`, `#`) and selecting the nine AI / data / software jobs.
>
> <details><summary>Version history</summary>
>
> - **v01** – Project set-up: introduction, loading the May 2023 BLS data, cleaning (`*`, `**`, `#`) and selecting the nine AI / data / software jobs.
>
> </details>

In [1]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

# Load dataset (keep this notebook in the same folder as the Excel file – loading takes 1-2 minutes)
df = pd.read_excel("all_data_M_2023.xlsx")
df.head()

,AREA,AREA_TITLE,AREA_TYPE,PRIM_STATE,NAICS,NAICS_TITLE,I_GROUP,OWN_CODE,OCC_CODE,OCC_TITLE,O_GROUP,TOT_EMP,EMP_PRSE,JOBS_1000,LOC_QUOTIENT,PCT_TOTAL,PCT_RPT,H_MEAN,A_MEAN,MEAN_PRSE,H_PCT10,H_PCT25,H_MEDIAN,H_PCT75,H_PCT90,A_PCT10,A_PCT25,A_MEDIAN,A_PCT75,A_PCT90,ANNUAL,HOURLY
0,99,U.S.,1,US,000000,Cross-industry,cross-industry,1235,00-0000,All Occupations,total,151853870,0,NaN,NaN,NaN,NaN,31.48,65470,0.3,13.97,17.14,23.11,37.01,58.4,29050,35660,48060,76980,121470,NaN,NaN
1,99,U.S.,1,US,000000,Cross-industry,cross-industry,1235,11-0000,Management Occupations,major,10495770,0.4,NaN,NaN,NaN,NaN,66.23,137750,0.5,26.23,37.66,56.19,81.29,111.36,54550,78330,116880,169090,231620,NaN,NaN
2,99,U.S.,1,US,000000,Cross-industry,cross-industry,1235,11-1000,Top Executives,minor,3751510,0.4,NaN,NaN,NaN,NaN,65.43,136100,0.9,22.31,31.81,49.74,79.57,#,46400,66170,103460,165500,#,NaN,NaN
3,99,U.S.,1,US,000000,Cross-industry,cross-industry,1235,11-1010,Chief Executives,broad,211230,1.4,NaN,NaN,NaN,NaN,124.47,258900,0.7,38.46,62.9,99.37,#,#,80000,130840,206680,#,#,NaN,NaN
4,99,U.S.,1,US,000000,Cross-industry,cross-industry,1235,11-1011,Chief Executives,detailed,211230,1.4,NaN,NaN,NaN,NaN,124.47,258900,0.7,38.46,62.9,99.37,#,#,80000,130840,206680,#,#,NaN,NaN


### Cleaning the data
Some number columns contain symbols instead of numbers:
- `*` or `**` = the value was not published (too few survey answers), so we turn it into a missing value (`NaN`).
- `#` = the salary is **\$239,200 or more** (the highest value the BLS publishes), so we replace it with 239200.

In [2]:
number_columns = ["TOT_EMP", "JOBS_1000", "LOC_QUOTIENT", "A_MEDIAN", "A_PCT10", "A_PCT25", "A_PCT75", "A_PCT90"]

for col in number_columns:
    df[col] = df[col].replace("#", 239200)
    df[col] = pd.to_numeric(df[col], errors="coerce")   # '*' and '**' become NaN

# The nine AI / Data Science / Software Development jobs and the field each belongs to
fields = {
    "Computer and Information Research Scientists": "AI & Research",
    "Data Scientists": "Data Science",
    "Statisticians": "Data Science",
    "Operations Research Analysts": "Data Science",
    "Mathematicians": "Data Science",
    "Database Architects": "Data Science",
    "Software Developers": "Software Development",
    "Software Quality Assurance Analysts and Testers": "Software Development",
    "Web Developers": "Software Development",
}

# National data (AREA_TYPE 1 = whole U.S.), all industries together, detailed job level only
# (O_GROUP "detailed" avoids double counting - some jobs also appear as a "broad" summary row)
usa = df[(df["AREA_TYPE"] == 1) & (df["I_GROUP"] == "cross-industry") & (df["O_GROUP"] == "detailed")]

# Keep only our nine jobs and add the field column
tech = usa[usa["OCC_TITLE"].isin(fields.keys())].copy()
tech["Field"] = tech["OCC_TITLE"].map(fields)

tech[["OCC_TITLE", "Field", "TOT_EMP", "A_MEDIAN"]].sort_values("TOT_EMP", ascending=False)

,OCC_TITLE,Field,TOT_EMP,A_MEDIAN
149,Software Developers,Software Development,1656880.0,132270.0
150,Software Quality Assurance Analysts and Testers,Software Development,203040.0,101800.0
165,Data Scientists,Data Science,192710.0,108020.0
161,Operations Research Analysts,Data Science,117880.0,83640.0
151,Web Developers,Software Development,85350.0,84960.0
145,Database Architects,Data Science,59920.0,134700.0
138,Computer and Information Research Scientists,AI & Research,35210.0,145080.0
163,Statisticians,Data Science,29950.0,104110.0
159,Mathematicians,Data Science,2220.0,116440.0
